<a href="https://colab.research.google.com/github/columbia-data-club/meetings/blob/main/2025/april_02_data_engineering_with_polars_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![A blue background with the pandas logo and the words Columbia Data Club on it](https://raw.githubusercontent.com/columbia-data-club/meetings/main/assets/images/2025/polars.png)

# Python Data Engineering with Polars II

April 2, 2025

by [Moacir P. de Sá Pereira](https://moacir.com) for the [Columbia Data Club](https://github.com/columbia-data-club/)


This notebook builds on [an introduction to data engineering](https://github.com/columbia-data-club/meetings/blob/main/2025/march_5_data_engineering_with_polars_1.ipynb) with [Polars](https://pola.rs). A basic understanding of Python syntax (such as the one covered in the Data Club’s [Intro to Python video](https://youtu.be/l45rzo4MUHs)) should suffice.

We will continue looking to our perennial favorite today, [NYC Yellow Cab trip data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page).

## Fire up the Data

Last time, we broadly discussed the two Polars mental models for data engineering: **contexts** and **expressions**. As a reminder:

* An **expression** is a lazy representation of a data transformation. We will be focusing on those today.
* A **context** is, well, a context in which an expression is evaluated. We mostly concern ourselves with four of them:
  * `select` chooses a subset of columns to transform and/or return
  * `with_columns` appends columns to already existing columns
  * `filter` filters rows based on certain criteria
  * `group_by` groups rows based on certain criteria, typically for aggregating.

Importantly, because expressions are lazy, we can assign them to variables. But let’s load the data and have a look at the columns again.

In [ ]:
import polars as pl

df = pl.read_parquet("https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-12.parquet")
df.glimpse()

## Expressions

Last time we created an expression for deriving the tip percentage. Let’s do that again just to refresh what expressions look like.

In [ ]:
tip_pct = (pl.col("tip_amount") / (pl.col("fare_amount") + 0.001)).round(2)
df.select(
    "tip_amount",
    "fare_amount",
    tip_pct.alias("tip_pct")
).glimpse()

I added a fudge of 1/10th of a cent to the fare amount so that we would have no 0s in the denominator when calculating the tip percentage. In fact, a $0 fare was not the most peculiar result:

In [ ]:
df.with_columns(tip_pct.alias("tip_pct"))["tip_pct"].describe()

We can do comparisons pretty easily, too, and like last time we can make conditional expressions.

In [ ]:
big_spender = pl.when(pl.col("tip_amount") > 5).then(
    pl.lit("Big Spender")
  ).otherwise(
    pl.lit("Cheapskate")
  ).alias("spender_type")

df.select("tip_amount", big_spender).head()

We can also operate on multiple columns at once, and they can be selected using various criteria:

In [ ]:
df.select(pl.col(pl.Float64)).glimpse()

In [ ]:
money_columns = ["fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount", \
  "improvement_surcharge", "total_amount", "congestion_surcharge", "Airport_fee"]
dollar_to_euro = 0.90 # Not the real rate.

df.select(
    (pl.col(money_columns) * dollar_to_euro).round(2).name.suffix("_in_euro")
).glimpse()

In [ ]:
amounts = pl.col("^.*_amount$")
df.select(
    (amounts * dollar_to_euro).round(2).name.suffix("_in_euro")
).glimpse()

## Expressions Can Take Arguments? 🫣

Above we have used expressions rather conveniently as a kind of shorthand for constant transformations. But say we wanted to do something dynamic… like fetch the current exchange rate.

In [ ]:
import requests

def get_usd_to_eur():
  url = "https://api.frankfurter.app/latest?from=USD&to=EUR"
  try:
    response = requests.get(url)
    data = response.json()
    rate = data['rates']['EUR']
    return rate
  except Exception as e:
    print(f"Error fetching exchange rate: {e}")
    return None

print(get_usd_to_eur())

In [ ]:
def convert_to_euro(column): # Of course this is a slightly silly example
    return (pl.col(column) * get_usd_to_eur()).round(2).name.suffix("_in_euro")

df.select("fare_amount", convert_to_euro("fare_amount")).glimpse()

In [ ]:
def convert_column_list_to_euro(column_list):
  for column in column_list:
    yield convert_to_euro(column)

df.with_columns(
  convert_column_list_to_euro(money_columns)
).select(
  pl.col([f"{col}_in_euro" for col in money_columns])
).glimpse()

Polars has a bunch more selectors like this hidden in the submodule [`selectors`](https://docs.pola.rs/api/python/stable/reference/selectors.html), which can give even more fine-grained selection of columns. In short, the effort here is to think of data transformation outside the (to me, at least) common pandas paradigm of using `.apply()` with a lambda, or, even worse, using `.iterrows()`.

In fact, in the [Aggregation](https://docs.pola.rs/user-guide/expressions/aggregation/#do-not-kill-parallelization) section of the documentation, they write:

> Polars will try to parallelize the computation of the aggregating functions over the groups, so it is recommended that you avoid using `lambda`s and custom Python functions as much as possible. Instead, try to stay within the realm of the Polars expression API


Other things we can do in expressions:

* [cast](https://docs.pola.rs/user-guide/expressions/casting/) from one datatype to another
* work with strings using the [`str` namespace](https://docs.pola.rs/user-guide/expressions/casting/#converting-strings-to-numeric-data-types)
* work with columns that have [lists or arrays as their data type](https://docs.pola.rs/user-guide/expressions/casting/#converting-strings-to-numeric-data-types)
* work with [`Enum` (fixed vocabulary) and `Categorical` (unfixed vocabulary)](https://docs.pola.rs/user-guide/expressions/categorical-data-and-enums/) data types

## Folds

I'm not entirely sure I understand folds, but they are kind of like reductions, and the examples Polars gives are functions I have used in the past: `sum_horizontal`, for example, which sums a set of columns horizontally:

In [ ]:
money_columns = [col for col in money_columns if col != "total_amount"]

In [ ]:
df.with_columns(
  pl.sum_horizontal(pl.col(money_columns)).round(2).alias("calculated_total")
).select(
  (pl.col("total_amount") - pl.col("calculated_total")).alias("diff")
).describe()

Polars reimplements `sum_horizontal` as a fold, so we can see [how folds work](https://docs.pola.rs/user-guide/expressions/folds/):

In [ ]:
import operator

df.select(
  "total_amount",
  pl.fold(
    acc=pl.lit(0),
    function=operator.add,
    exprs=pl.col(money_columns)
  ).round(2).alias("calculated_total")
).glimpse()

## NumPy Universal Functions (ufuncs)

In the quest to avoid breaking parallelization to keep Polars fast, we should avoid using Python-specific code. What if we need functions from the `math` library, though? Use instead a [NumPy universal function](https://numpy.org/doc/stable/reference/ufuncs.html#available-ufuncs). Here, we consider a column in Polars (a Polars `Series` type) to be a numpy array. This could break because of missing data, in which case, cast the Series explicitly to a NumPy array.

In [ ]:
import numpy as np

df = pl.DataFrame(
  {
    "a": [1, 2, 3],
    "b": [4, None, 6]
  }
)

df

In [ ]:
df = df.with_columns(np.log(pl.all()).name.suffix("_log"))
df

In [ ]:
b = df["b"].to_numpy()
b

In [ ]:
b_df = pl.DataFrame({"b_numpy_log": np.log(b)})
df = pl.concat([df, b_df], how="horizontal")
df